In [11]:
# %%
# SINGLE-DATE NEURAL ODE FIT (no state process x)
# du/dtau = f(tau, u) ; fit a single discount curve at target date
from dataclasses import dataclass
from typing import Optional, Union
import pandas as pd
import numpy as np
import torch
from torch import nn
from torchdiffeq import odeint
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Torch defaults
torch.set_default_dtype(torch.float32)
torch.set_default_device('cpu')
torch.manual_seed(42)

# --- ODE integration step for u-ODE
ODE_STEP = 1/4  # fixed step for RK4 over τ

# =====================================================================================
# DATA LOADING (as before)
# =====================================================================================
discounts = pd.read_excel('discounts.xlsx', index_col='date').dropna()
discounts = discounts
panel_data = pd.read_excel('macrodata.xlsx', index_col='meeting_date').dropna()

tenors_np          = np.asarray(discounts.columns[1:], dtype=float)  # M tenors
obs_discounts_np   = discounts.to_numpy()
discount_curves_np = obs_discounts_np[:, 1:]
short_rate_np      = ((1.0 / obs_discounts_np[:, 0]) - 1.0) * 360.0   # simple money market
times_np           = np.asarray((discounts.index - discounts.index[0]).days / 360.0)

meeting_dates      = pd.to_datetime(panel_data.index)
meeting_times_np   = np.asarray((meeting_dates - discounts.index[0]).days / 360.0)

# --- to torch
tenors         = torch.tensor(tenors_np, dtype=torch.float32)                  # (M,)
discount_curves= torch.tensor(discount_curves_np, dtype=torch.float32)         # (N, M)
short_rate     = torch.tensor(short_rate_np, dtype=torch.float32)              # (N,)
meeting_times  = torch.tensor(meeting_times_np, dtype=torch.float32)           # (K_meetings_all,)
times          = torch.tensor(times_np, dtype=torch.float32)                   # (N,) 

In [12]:
class MLP(nn.Module):
    def __init__(self, n_input: int, n_hidden: int, n_out: int, n_layers: int = 2, activation=nn.Tanh):
        super().__init__()
        layers = [nn.Linear(n_input, n_hidden), activation()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_hidden, n_hidden), activation()]
        layers += [nn.Linear(n_hidden, n_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class TauU_ODE(nn.Module):
    def __init__(self, net: nn.Module):
        super().__init__()
        self.net = net

    def forward(self, tau, u):
        # tau: scalar; u: scalar or (...,)
        if tau.dim() == 0:
            tau_ = tau.reshape(1, 1)   # (1,1)
            u_   = u.view(1, -1)       # (1,1)
        else:
            tau_ = tau.view(-1, 1)
            u_   = u.view(-1, 1)
        inp = torch.cat([tau_, u_], dim=1)  # (B,2)
        return self.net(inp)                # (B,1)

def init_mlp_weights(model: nn.Module, output_scale: float = 1e-2):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if m.out_features == 1:
                nn.init.uniform_(m.weight, -output_scale, output_scale)
                nn.init.zeros_(m.bias)
            else:
                nn.init.orthogonal_(m.weight, gain=0.5)
                nn.init.zeros_(m.bias)

class NeuralDiscounts(nn.Module):
    def __init__(
        self,
        u_ode: TauU_ODE,
        possible_jumps: torch.Tensor,          # (J,)
        jump_size: float,                      # Δ for logits scaling
        absolute_meeting_times: torch.Tensor,  # all calendar meetings (years)
        obs_meetings: int = 12,
        meeting_freq: torch.Tensor = torch.tensor([1/12], dtype=torch.float32)
    ):
        super().__init__()
        self.u_ode = u_ode
        self.possible_jumps = possible_jumps.float()
        self.jump_size = float(jump_size)
        self.absolute_meeting_times = absolute_meeting_times.float()
        self.obs_meetings = int(obs_meetings)
        self.meeting_freq = meeting_freq.float()

    def generate_meeting_times(self, t: torch.Tensor, T: torch.Tensor):
        t = float(t.squeeze())
        T = float(T.squeeze())
        freq = float(self.meeting_freq.item())

        rel = (self.absolute_meeting_times - t)      # τ from t
        mask = (rel > 0.0) & (rel <= T)
        real_tau = rel[mask]
        real_tau, _ = torch.sort(real_tau)
        if real_tau.numel() > self.obs_meetings:
            real_tau = real_tau[: self.obs_meetings]

        if real_tau.numel() == 0:
            start = min(freq, T + 1e-12)
            if start >= T - 1e-12:
                return torch.empty(0)
            return torch.arange(start, T + 1e-12, freq)

        last_real = float(real_tau[-1].item())
        start = last_real + freq
        if start > T - 1e-12:
            return real_tau

        ext = torch.arange(start, T + 1e-12, freq)
        all_tau = torch.cat([real_tau.to(ext.dtype), ext], dim=0)
        all_tau = torch.unique(all_tau)
        all_tau, _ = torch.sort(all_tau)
        return all_tau

    def jump_probs(self, u: torch.Tensor) -> torch.Tensor:
        if u.dim() == 1:
            u = u.unsqueeze(-1)  # (K,1)
        logits = (u / self.jump_size) * self.possible_jumps.view(1, -1)  # (K,J)
        logits = logits - logits.amax(dim=-1, keepdim=True)
        return torch.softmax(logits, dim=-1)

    def solve_u(self, tau_grid: torch.Tensor, u0: Optional[torch.Tensor] = None) -> torch.Tensor:
        if tau_grid.numel() == 0:
            return tau_grid
        if u0 is None:
            u0 = torch.zeros(1)  # u(0)=0
        path = odeint(self.u_ode, u0, tau_grid, method='rk4', options={'step_size': ODE_STEP})
        return path.squeeze(1)  # (K,)

    def forward_discounts(self, t: torch.Tensor, tau_grid: torch.Tensor, deltas_out: bool = False):
        if tau_grid.numel() == 0:
            return (torch.empty(0), torch.empty(0)) if deltas_out else torch.empty(0)

        deltas = torch.empty_like(tau_grid)
        deltas[0]  = tau_grid[0]
        deltas[1:] = tau_grid[1:] - tau_grid[:-1]

        u = self.solve_u(tau_grid)  # (K,)
        probs = self.jump_probs(u)  # (K,J)
        djs = torch.exp(-self.possible_jumps.view(1, -1) * deltas.view(-1, 1))  # (K,J)

        fwd_expected = (djs * probs).sum(dim=1)  # (K,)
        return (fwd_expected, deltas) if deltas_out else fwd_expected

    def discount(self, t: torch.Tensor, T: torch.Tensor, r0: torch.Tensor) -> torch.Tensor:
        base = torch.exp(-r0 * T)
        tau_grid = self.generate_meeting_times(t, T)
        if tau_grid.numel() == 0:
            return base
        fwd_expected, deltas = self.forward_discounts(t, tau_grid, deltas_out=True)
        # D_clean(T) = Π_k fwd_expected_k ; D(T) = exp(-r0 T) * D_clean(T)
        return base * fwd_expected.prod()

    def discount_curve(self, t: torch.Tensor, T_vec: torch.Tensor, r0: torch.Tensor) -> torch.Tensor:
        base = torch.exp(-r0 * T_vec)
        out = []
        for T in T_vec:
            tau_grid = self.generate_meeting_times(t, T)
            if tau_grid.numel() == 0:
                out.append(torch.exp(-r0 * T))
            else:
                fwd_expected = self.forward_discounts(t, tau_grid)
                out.append(torch.exp(-r0 * T) * fwd_expected.prod())
        return torch.stack(out)

In [13]:
def interp1d_clamped(xq: torch.Tensor, xp: torch.Tensor, fp: torch.Tensor) -> torch.Tensor:
    assert xp.dim() == 1 and fp.dim() == 1
    xq = xq.to(xp.device)
    out = torch.empty_like(xq)
    left  = xq <= xp[0]
    right = xq >= xp[-1]
    mid   = (~left) & (~right)
    out[left]  = fp[0]
    out[right] = fp[-1]
    if mid.any():
        xm = xq[mid]
        idx = torch.searchsorted(xp, xm, right=False).clamp(1, xp.numel()-1)
        x0 = xp[idx-1]; x1 = xp[idx]
        y0 = fp[idx-1]; y1 = fp[idx]
        w  = (xm - x0) / (x1 - x0 + 1e-12)
        out[mid] = y0 + w*(y1 - y0)
    return out

In [14]:
target_idx: int = 4000
t0   = times[target_idx]              
r0   = short_rate[target_idx]         
Dobs = discount_curves[target_idx]     
T_max = tenors[-1]

possible_jumps = torch.tensor([-50, -25, 0.0, 25, 50], dtype=torch.float32) / 10_000
jump_size = 25 / 10_000
absolute_meeting_times = meeting_times

ode_net = MLP(n_input=2, n_hidden=10, n_out=1, n_layers=2, activation=nn.Tanh)
init_mlp_weights(ode_net, output_scale=1e-2)
u_ode = TauU_ODE(ode_net)

nd = NeuralDiscounts(
    u_ode=u_ode,
    possible_jumps=possible_jumps,
    jump_size=jump_size,
    absolute_meeting_times=absolute_meeting_times,
    obs_meetings=12,
    meeting_freq=torch.tensor([1/4], dtype=torch.float32)  # quarterly padding if needed
)

with torch.no_grad():
    tau_grid = nd.generate_meeting_times(t0, T_max)  # (K,)
K = int(tau_grid.numel())

deltas = torch.empty_like(tau_grid)
deltas[0]  = tau_grid[0]
deltas[1:] = tau_grid[1:] - tau_grid[:-1]

# Interpolate the observed discount curve to τ_k points
D_tau = interp1d_clamped(tau_grid, tenors, Dobs).view(-1)        
D_tau_prev = torch.cat([tau_grid.new_ones(1), D_tau[:-1]], dim=0) 

per_seg_obs = (D_tau / D_tau_prev)
fwd_clean_obs = per_seg_obs / torch.exp(-r0 * deltas)
fwd_clean_obs = fwd_clean_obs

def train_forward_segments(
    *,
    model: NeuralDiscounts,
    t0: torch.Tensor,
    tau_grid: torch.Tensor,
    deltas: torch.Tensor,
    fwd_clean_obs: torch.Tensor,
    epochs: int = 3000,
    lr: float = 2e-3,
    l2_reg: float = 0.0
):
    params = list(model.u_ode.parameters())
    opt = torch.optim.Adam(params, lr=lr)

    for ep in range(1, epochs+1):
        opt.zero_grad()
        # model forward (clean) per segment
        fwd_model = model.forward_discounts(t0, tau_grid)  # (K,)
        loss_mse = torch.mean((fwd_model - fwd_clean_obs)**2)
        loss = loss_mse

        if l2_reg > 0.0:
            l2 = sum((p**2).sum() for p in params)
            loss = loss + l2_reg * l2

        loss.backward()
        opt.step()

        mae = torch.mean(torch.abs(fwd_model - fwd_clean_obs)).item()
        print(f"epoch {ep:5d} | forward-MSE {loss_mse.item():.6e} | MAE {mae:.6e}")

    return model

_ = train_forward_segments(
    model=nd,
    t0=t0,
    tau_grid=tau_grid,
    deltas=deltas,
    fwd_clean_obs=fwd_clean_obs,
    epochs=10_000,
    lr=1e-3,
    l2_reg=0.0
)

epoch     1 | forward-MSE 8.255740e-05 | MAE 8.187669e-03
epoch     2 | forward-MSE 8.232646e-05 | MAE 8.171680e-03
epoch     3 | forward-MSE 8.209712e-05 | MAE 8.155730e-03
epoch     4 | forward-MSE 8.186886e-05 | MAE 8.139799e-03
epoch     5 | forward-MSE 8.164164e-05 | MAE 8.123892e-03
epoch     6 | forward-MSE 8.141558e-05 | MAE 8.108012e-03
epoch     7 | forward-MSE 8.119013e-05 | MAE 8.092118e-03
epoch     8 | forward-MSE 8.096576e-05 | MAE 8.076253e-03
epoch     9 | forward-MSE 8.074137e-05 | MAE 8.060344e-03
epoch    10 | forward-MSE 8.051721e-05 | MAE 8.044398e-03
epoch    11 | forward-MSE 8.029251e-05 | MAE 8.028365e-03
epoch    12 | forward-MSE 8.006812e-05 | MAE 8.012315e-03
epoch    13 | forward-MSE 7.984291e-05 | MAE 7.996168e-03
epoch    14 | forward-MSE 7.961742e-05 | MAE 7.979963e-03
epoch    15 | forward-MSE 7.939128e-05 | MAE 7.963673e-03
epoch    16 | forward-MSE 7.916494e-05 | MAE 7.947339e-03
epoch    17 | forward-MSE 7.893881e-05 | MAE 7.930978e-03
epoch    18 | 

In [16]:
# =============================================================================
# DIAGNOSTICS & PLOTS
# =============================================================================
# 1) Compare forward *clean* factors (model vs obs) on the τ grid
with torch.no_grad():
    fwd_model = nd.forward_discounts(t0, tau_grid)       # clean per-segment discount
    base_fwd  = torch.exp(-r0 * deltas)                  # baseline per-segment discount
    per_seg_disc = base_fwd * fwd_model                  # actual per-segment discount in (0,1]
    
    # before (wrong): D_model = torch.cumprod(1.0 / per_seg_disc, dim=0)
    # after (correct):
    D_model = torch.cumprod(per_seg_disc, dim=0)         # D(0)=1 implicit
    D_model = D_model

print("\nSummary (forwards):")
print(f"  Forward RMSE (clean): {torch.sqrt(((fwd_model - fwd_clean_obs)**2).mean()).item():.6e}")
print(f"  Forward MAE  (clean): {torch.mean(torch.abs(fwd_model - fwd_clean_obs)).item():.6e}")

# 2) Interpolate obs curve to τ grid to compare discounts on grid
D_tau = interp1d_clamped(tau_grid, tenors, Dobs).view(-1)

# 3) For plotting also build a smooth curve fit across all input tenors
with torch.no_grad():
    Dpred_full = nd.discount_curve(t0, tenors, r0)

# --- Plot A: Clean forward factors per segment
figA = go.Figure()
figA.add_trace(go.Scatter(x=tau_grid.numpy(), y=fwd_clean_obs.numpy(), mode='markers', name='Obs clean forward'))
figA.add_trace(go.Scatter(x=tau_grid.numpy(), y=fwd_model.detach().numpy(), mode='lines+markers', name='Model clean forward'))
figA.update_layout(title='Per-segment CLEAN forward discount factors',
                   xaxis_title='τ (years from t0) at segment end',
                   yaxis_title='E[exp(-J Δτ)]',
                   template='plotly_white')
figA.show()

# --- Plot B: Discount factors on τ-grid (obs via interp vs model reconstructed)
figB = go.Figure()
figB.add_trace(go.Scatter(x=tau_grid.numpy(), y=D_tau.numpy(), mode='markers', name='Observed D(t0, τ_k)'))
figB.add_trace(go.Scatter(x=tau_grid.numpy(), y=D_model.detach().numpy(), mode='lines+markers', name='Model D(t0, τ_k)'))
figB.update_layout(title='Discounts on meeting grid',
                   xaxis_title='τ (years from t0)',
                   yaxis_title='Discount factor',
                   template='plotly_white')
figB.show()

# --- Plot C: Full curve across input tenors
figC = go.Figure()
figC.add_trace(go.Scatter(x=tenors_np, y=Dobs.numpy(), mode='markers', name='Observed D(t0,T)'))
figC.add_trace(go.Scatter(x=tenors_np, y=Dpred_full.detach().numpy(), mode='lines', name='Model D(t0,T)'))
figC.update_layout(title=f"Full Curve Fit @ idx {int(target_idx)} (trained on forwards)",
                   xaxis_title='Maturity (years)', yaxis_title='Discount factor',
                   template='plotly_white')
figC.show()

# --- Plot D: u(τ) + meeting markers + jump probs
with torch.no_grad():
    T_ref = tenors[len(tenors)//2]
    tau_dense = torch.linspace(0.0, float(T_ref.item()), 400)
    u0 = torch.zeros(1)
    u_dense = odeint(nd.u_ode, u0, tau_dense, method='rk4', options={'step_size': ODE_STEP}).squeeze()

    tau_meet = nd.generate_meeting_times(t0, T_ref)
    if tau_meet.numel() > 0:
        u_meet = nd.solve_u(tau_meet)
        probs_meet = nd.jump_probs(u_meet).detach().numpy()
        tau_meet_np = tau_meet.detach().numpy()
    else:
        u_meet = torch.tensor([])
        probs_meet = None
        tau_meet_np = np.array([])

rows = 2 if (tau_meet.numel() > 0) else 1
figD = make_subplots(rows=rows, cols=1, shared_xaxes=True,
                     subplot_titles=(["u(τ) shape", "Jump probabilities at meeting times"]
                                     if rows==2 else ["u(τ) shape"]))
figD.add_trace(go.Scatter(x=tau_dense.numpy(), y=u_dense.detach().numpy(), mode='lines', name='u(τ) dense'),
               row=1, col=1)
if rows == 2:
    figD.add_trace(go.Scatter(x=tau_meet_np, y=u_meet.detach().numpy(), mode='markers', name='u(τ) @ meetings'),
                   row=1, col=1)
    for tm in tau_meet_np:
        figD.add_shape(type="line", x0=tm, x1=tm,
                       y0=float(min(u_dense.min(), u_meet.min())),
                       y1=float(max(u_dense.max(), u_meet.max())),
                       line=dict(width=1, dash="dot"), row=1, col=1)
    jump_vals_bp = (nd.possible_jumps * 10_000).detach().numpy()
    for j in range(probs_meet.shape[1]):
        figD.add_trace(go.Scatter(x=tau_meet_np, y=probs_meet[:, j], mode='lines+markers',
                                  name=f'P(J={jump_vals_bp[j]:.0f} bp)'), row=2, col=1)
    figD.update_yaxes(title_text="Probability", range=[0,1], row=2, col=1)
figD.update_yaxes(title_text="u(τ)", row=1, col=1)
figD.update_xaxes(title_text="τ (years from t0)", row=rows, col=1)
figD.update_layout(title="u(τ) & Jump probabilities", template="plotly_white", height=700)
figD.show()

# --- Quick print at a few tenors
with torch.no_grad():
    sample_idx = [0, min(5, len(tenors)-1), min(10, len(tenors)-1), len(tenors)-1]
    for i in sample_idx:
        T = tenors[i]
        Dm = nd.discount(t0, T, r0)
        print(f"T={float(T.item()):5.2f} | D_obs={float(Dobs[i].item()):.6f} | D_pred={float(Dm.item()):.6f}")



Summary (forwards):
  Forward RMSE (clean): 8.178744e-03
  Forward MAE  (clean): 7.115266e-03


T= 0.50 | D_obs=0.969779 | D_pred=0.961407
T= 3.00 | D_obs=0.872022 | D_pred=0.792112
T= 5.50 | D_obs=0.773297 | D_pred=0.652592
T=10.00 | D_obs=0.601977 | D_pred=0.460449
